# Prática — SVM

**Problema:** prever quem sobreviveu ao **Titanic**.
Comparando o SVM com a Random Forest, que você mesmo(a) vai montar.


Rode as células **uma por uma, de cima para baixo** (`Shift + Enter`). As células com `___` só funcionam depois que você completar.

## 0. Preparação (já pronta, só rode)

In [3]:
import pandas as pd
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(url)

# Limpeza e encoding (iguais à prática anterior)
df = df[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]].copy()
df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna("S")
df["sex"] = df["sex"].map({"male": 0, "female": 1})
df = pd.get_dummies(df, columns=["embarked"], dtype=int)

X = df.drop(columns="survived")
y = df["survived"]
print(f"{len(df)} passageiros | {y.mean():.0%} sobreviveram")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultados = []

def avaliar(nome, modelo):
    r = cross_validate(modelo, X, y, cv=cv, return_train_score=True)
    treino, valid = r["train_score"].mean(), r["test_score"].mean()
    resultados.append({"Modelo": nome, "Treino": treino, "Validação": valid})
    print(f"{nome:<40} treino: {treino:.1%}   validação: {valid:.1%}")

891 passageiros | 38% sobreviveram


---
## Etapa 0 — Sua Random Forest de referência

Antes do SVM, monte **você** a Random Forest da aula passada. Ela será a referência para comparar.

> **Lembrete:** a Random Forest treina muitas árvores diferentes e decide pelo voto da maioria.
> Hiperparâmetros principais: `n_estimators` (nº de árvores) e `max_depth` (profundidade de cada árvore).
> A classe se chama `RandomForestClassifier` (já importada).

**0.1** Escreva a linha completa do modelo: **300 árvores** e **profundidade máxima 6**.

In [4]:
avaliar("Random Forest (referência)", RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42))

Random Forest (referência)               treino: 87.2%   validação: 83.3%


<details><summary> Travou? Clique para ver uma dica</summary>

O formato é `NomeDoModelo(parametro=valor, parametro=valor)`. Use `n_estimators=300`, `max_depth=6`
e acrescente `random_state=42` para o resultado ser reproduzível.
</details>

<details><summary>Ainda travado? Segunda dica</summary>

`RandomForestClassifier(n_estimators=300, max_depth=___, random_state=42)`: falta só um número.
</details>

---
## Etapa 1 — O SVM e a escala dos dados

> **Lembrete:** o SVM traça a fronteira com a **maior margem** entre as classes e decide por **distância**.
> Se uma coluna tem números muito maiores que as outras, ela domina a distância.
> O `StandardScaler` coloca todas as colunas na mesma escala, e o `make_pipeline` encadeia escala + modelo.

**1.1** Veja as escalas das colunas:

In [5]:
X.describe().loc[["min", "max"]]

,pclass,sex,age,sibsp,parch,fare,embarked_C,embarked_Q,embarked_S
min,1.0,0.0,0.42,0.0,0.0,0.0000,0.0,0.0,0.0
max,3.0,1.0,80.00,8.0,6.0,512.3292,1.0,1.0,1.0


**1.2** Rode o SVM **sem** escalar os dados:

In [6]:
avaliar("SVM sem escala", SVC())

SVM sem escala                           treino: 68.2%   validação: 67.8%


**1.3** Agora com escala. Complete o pipeline com o `StandardScaler()`.
Dica: na demo foi `make_pipeline(StandardScaler(), SVC())`.

In [7]:
avaliar("SVM com escala", make_pipeline(StandardScaler(), SVC()))

SVM com escala                           treino: 84.1%   validação: 82.7%


**1.4 Quanto a validação mudou? Qual coluna você acha que atrapalhava o SVM sem escala, e por quê?**

*Sua resposta:* A validação subiu de 67,8% para 82,7%, ou seja, cerca de 15 pontos a mais, só por escalar os dados.

Acho que a coluna que atrapalhava era a **fare** (tarifa), porque vai até 512 enquanto as outras ficam entre 0 e 8. Como o SVM decide por distância, a `fare` dominava o cálculo e colunas importantes como `sex` quase não influenciavam. Depois do `StandardScaler`, todas passaram a ter o mesmo peso e o modelo melhorou.

---
## Etapa 2 — O hiperparâmetro C

> **Lembrete:** `C` controla o quanto o SVM aceita errar no treino.
> **C baixo** → margem larga, tolera erros, modelo simples.
> **C alto** → margem estreita, tenta acertar tudo, risco de decorar (overfitting).
> O padrão é `C=1`.

**2.1** Teste C muito baixo e muito alto. Substitua por `0.01` e `100`:

In [8]:
avaliar("SVM C=0.01", make_pipeline(StandardScaler(), SVC(C=0.01)))
avaliar("SVM C=100",  make_pipeline(StandardScaler(), SVC(C=100)))

SVM C=0.01                               treino: 61.6%   validação: 61.6%
SVM C=100                                treino: 88.0%   validação: 79.8%


**2.2 Com C=0.01, a validação ficou perto de 62%. Que número é esse? (Dica: quantos passageiros morreram?)**

*Sua resposta:* Esse número é a proporção de passageiros que **morreram**. No começo do código, a saída mostrou que cerca de 38% sobreviveram, então 62% não sobreviveram.

Com C=0.01, o modelo ficou simples demais e acabou chutando "ninguém sobrevive" para todo mundo. Por isso ele acerta exatamente a fração de quem morreu, ~62%. Ou seja, não aprendeu nenhum padrão (como `sex` ou `pclass`), é underfitting.

**2.3 Com C=100, o que aconteceu com a distância entre treino e validação?**

*Sua resposta:* A distância entre treino e validação aumentou bastante. Com C=100, o treino foi para 88,0% e a validação caiu para 79,8%, um gap de **8,2 pontos**, contra 1,4 pontos com C=1.

Isso acontece porque o C alto faz o SVM tentar acertar quase todos os pontos do treino, com uma margem estreita. Ele acaba decorando detalhes que não se repetem em passageiros novos, então a validação piora. É overfitting.

---
## Etapa 3 — Kernel e gamma

> **Lembrete:** o **kernel** permite fronteiras curvas. `kernel="linear"` só traça retas (planos);
> `kernel="rbf"` (o padrão) faz curvas, como no exemplo dos círculos.
> **`gamma`** controla o alcance de cada ponto: gamma alto → fronteira tortuosa, com "ilhas" → overfitting.

**3.1** Compare o kernel linear com o rbf. Complete com `"linear"`:

In [9]:
avaliar("SVM kernel linear", make_pipeline(StandardScaler(), SVC(kernel="linear")))

SVM kernel linear                        treino: 79.0%   validação: 78.7%


**3.2** Agora teste um gamma alto. Substitua por `10`:

In [10]:
avaliar("SVM gamma=10", make_pipeline(StandardScaler(), SVC(gamma=10)))

SVM gamma=10                             treino: 91.2%   validação: 75.0%


**3.3 O kernel rbf (Etapa 1.3) foi melhor que o linear? O que isso sugere sobre a fronteira entre sobreviventes e não sobreviventes?**

*Sua resposta:*

**3.4 Com gamma=10, o que aconteceu com treino e validação?**

*Sua resposta:* Sim, o rbf foi melhor: 82,7% de validação contra 78,7% do linear, uns 4 pontos a mais.

Isso sugere que a fronteira entre sobreviventes e não sobreviventes não é uma reta simples. Existem interações entre as colunas (por exemplo, o efeito da classe é diferente para homens e mulheres) que só uma fronteira curva consegue capturar.

---
## 🏆 Desafio — Busca automática de hiperparâmetros

Testar combinações na mão cansa. O `GridSearchCV` testa **todas** as combinações de uma grade
e devolve a melhor, usando validação cruzada.

Complete a grade com pelo menos 3 valores de `C` e 3 de `gamma`. Sugestão: `[0.1, 1, 10]` e `[0.01, 0.1, 1]`.
Atenção ao nome: dentro do pipeline, o parâmetro se chama `svc__C` (dois underlines).

In [11]:
from sklearn.model_selection import GridSearchCV

grade = {
    "svc__C":     [0.1, 1, 10],
    "svc__gamma": [0.01, 0.1, 1],
}
busca = GridSearchCV(make_pipeline(StandardScaler(), SVC()), grade, cv=cv)
busca.fit(X, y)

print("Melhor combinação:", busca.best_params_)
print(f"Validação: {busca.best_score_:.1%}")

Melhor combinação: {'svc__C': 1, 'svc__gamma': 0.1}
Validação: 82.5%


**Desafio: o melhor SVM superou a sua Random Forest (Etapa 0)? Por quanto?**

*Sua resposta:* Não, o melhor SVM não superou a Random Forest. O melhor SVM (GridSearch, C=1 e gamma=0.1) teve 82,5% de validação, contra 83,3% da Random Forest, uma diferença de **0,8 ponto percentual** a favor da floresta.

Essa diferença é pequena e provavelmente está dentro da variação normal entre folds, então na prática os dois modelos empatam nesse dataset. O SVM tem a vantagem de generalizar de forma mais estável (gap menor entre treino e validação), mas só funciona bem se os dados forem escalados.

---
## Seus resultados

In [13]:
tabela = (pd.DataFrame(resultados)
          .drop_duplicates("Modelo", keep="last")
          .sort_values("Validação", ascending=False)
          .reset_index(drop=True))

tabela["Treino"] = tabela["Treino"].map("{:.1%}".format)
tabela["Validação"] = tabela["Validação"].map("{:.1%}".format)
tabela

,Modelo,Treino,Validação
0,Random Forest (referência),87.2%,83.3%
1,SVM com escala,84.1%,82.7%
2,SVM C=100,88.0%,79.8%
3,SVM kernel linear,79.0%,78.7%
4,SVM gamma=10,91.2%,75.0%
5,SVM sem escala,68.2%,67.8%
6,SVM C=0.01,61.6%,61.6%


**Se você fosse colocar um destes modelos em produção, qual escolheria? Justifique em uma frase.**

*Sua resposta:* Eu escolheria a **Random Forest**, porque teve a maior validação (83,3%) e não depende de escalar os dados, o que deixa o modelo mais simples e menos sujeito a erro em produção. O SVM com escala empata na prática (82,7%) e generaliza de forma um pouco mais estável, então também seria uma boa opção.

Uma observação sobre a célula: a tabela só vai mostrar os modelos que você rodou com `avaliar`. A busca do `GridSearchCV` não passa por essa função, então ela não aparece na tabela.